In [ ]:

import sys
import time
import warnings
from datetime import datetime, timezone
from typing import Optional

warnings.filterwarnings("ignore")

# ── Dependency check ──────────────────────────────────────────────────────────
REQUIRED = {
    "huggingface_hub": "huggingface_hub",
    "sentence_transformers": "sentence-transformers",
    "pandas": "pandas",
    "numpy": "numpy",
    "sklearn": "scikit-learn",
    "rich": "rich",
}

missing = []
for module, package in REQUIRED.items():
    try:
        __import__(module)
    except ImportError:
        missing.append(package)

if missing:
    print(f"[ERROR] Missing packages. Run:\n  pip install {' '.join(missing)}")
    sys.exit(1)

import numpy as np
import pandas as pd
from huggingface_hub import HfApi, whoami
from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich import print as rprint
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

console = Console()

# Insert Access Token below
# user = whoami("ACCESS TOKEN")

# Keywords to pre-filter models via the HF API (cheap, no embeddings needed)
KEYWORD_FILTERS = [
    "vla", "vision-language-action",
    "robotics", "robot"
]

# Frontier topic sentences we embed and use for semantic similarity scoring
FRONTIER_TOPICS = [
    "vision language action model for robot manipulation",
    "video action model for general purpose robotics",
    "open claw robotic arm control with language model",
    "model quantization and inference optimization LLM",
    "weight pruning and compression for transformer models",
    "agentic coding assistant that can write and execute code autonomously",
    "large language model fine-tuned for software engineering agents",
]

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"  # ~80MB, fast
SIMILARITY_THRESHOLD = 0.25   # models above this score are considered "frontier"
MAX_MODELS_TO_FETCH = 200     # keep the POC quick; production would use more
ADOPTION_TOP_PERCENTILE = 25  # label = 1 if downloads in top 25% of cohort


# ── Step 1: Fetch models from HF Hub ─────────────────────────────────────────

def fetch_models(limit: int = MAX_MODELS_TO_FETCH) -> list[dict]:
    """
    Pull recently modified models that match at least one frontier keyword.
    Returns a list of raw metadata dicts.
    """
    api = HfApi()
    raw_models = []

    console.log(f"[cyan]Querying HF Hub API for up to {limit} recent frontier models…[/cyan]")

    for keyword in KEYWORD_FILTERS[:6]:  # limit keywords for POC speed
        try:
            results = list(api.list_models(
                sort="downloads",
                limit=100,
                cardData=True,       # fetch model card metadata
                pipeline_tag="robotics",
                fetch_config=False,
            ))
            raw_models.extend(results)
            console.log(f"  keyword=[bold]{keyword:<30}[/bold] → {len(results)} models")
        except Exception as e:
            console.log(f"  [yellow]Warning: keyword '{keyword}' failed: {e}[/yellow]")

    # Deduplicate by model id
    seen, unique = set(), []
    for m in raw_models:
        if m.modelId not in seen:
            seen.add(m.modelId)
            unique.append(m)

    console.log(f"[green]✓ Fetched {len(unique)} unique models after deduplication[/green]\n")
    return unique[:limit]


# ── Step 2: Build feature rows ────────────────────────────────────────────────

def hours_since(dt: Optional[datetime]) -> float:
    """Return hours elapsed since a given datetime (UTC). Returns -1 if None."""
    if dt is None:
        return -1.0
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    delta = datetime.now(timezone.utc) - dt
    return max(delta.total_seconds() / 3600, 0.0)


def extract_model_card_text(model_info) -> str:
    """Extract a plain-text snippet from model card metadata for embedding."""
    parts = []
    # model id itself is often descriptive
    parts.append(model_info.modelId.replace("/", " ").replace("-", " "))
    # tags
    if model_info.tags:
        parts.append(" ".join(model_info.tags[:30]))
    # pipeline tag
    if hasattr(model_info, "pipeline_tag") and model_info.pipeline_tag:
        parts.append(model_info.pipeline_tag)
    # card data description
    if model_info.cardData:
        cd = model_info.cardData
        if isinstance(cd, dict):
            for key in ("language", "license", "base_model", "datasets"):
                val = cd.get(key, "")
                if val:
                    parts.append(str(val)[:100])
    return " ".join(parts)[:512]  # cap at 512 chars; MiniLM handles ~256 tokens


def build_features(models: list) -> pd.DataFrame:
    """Compute metadata and velocity features for each model."""
    rows = []
    now = datetime.now(timezone.utc)

    for m in models:
        age_h = hours_since(m.created_at)
        modified_h = hours_since(m.lastModified)
        downloads = getattr(m, "downloads", 0) or 0
        likes = getattr(m, "likes", 0) or 0
        trending = getattr(m, "trendingScore", 0.0) or 0.0

        # Early velocity: downloads per hour since creation
        dl_per_hour = (downloads / age_h) if age_h > 0 else 0.0
        likes_per_hour = (likes / age_h) if age_h > 0 else 0.0

        # Binary flags
        created_dt = m.created_at
        if created_dt and created_dt.tzinfo is None:
            created_dt = created_dt.replace(tzinfo=timezone.utc)
        weekend_upload = int(created_dt.weekday() >= 5) if created_dt else 0

        library = getattr(m, "library_name", None) or ""

        rows.append({
            "model_id":        m.modelId,
            "age_hours":       round(age_h, 1),
            "modified_hours":  round(modified_h, 1),
            "downloads":       downloads,
            "likes":           likes,
            "trending_score":  round(trending, 4),
            "dl_per_hour":     round(dl_per_hour, 4),
            "likes_per_hour":  round(likes_per_hour, 4),
            "weekend_upload":  weekend_upload,
            "library":         library,
            "card_text":       extract_model_card_text(m),
        })

    return pd.DataFrame(rows)


# ── Step 3: Embed & score frontier similarity ─────────────────────────────────

def compute_frontier_scores(df: pd.DataFrame) -> pd.DataFrame:
    """
    Embed model card texts and frontier topic sentences.
    Add a frontier_score column = max cosine similarity across all topic vectors.
    """
    console.log(f"[cyan]Loading embedding model: {EMBEDDING_MODEL}…[/cyan]")
    t0 = time.time()
    embedder = SentenceTransformer(EMBEDDING_MODEL)
    console.log(f"  Loaded in {time.time()-t0:.1f}s")

    console.log("[cyan]Embedding frontier topic vectors…[/cyan]")
    topic_vecs = embedder.encode(FRONTIER_TOPICS, show_progress_bar=False)

    console.log(f"[cyan]Embedding {len(df)} model card texts…[/cyan]")
    t1 = time.time()
    card_vecs = embedder.encode(df["card_text"].tolist(), show_progress_bar=True,
                                batch_size=64)
    console.log(f"  Embedded in {time.time()-t1:.1f}s")

    # Shape: (n_models, n_topics) → take row-wise max
    sim_matrix = cosine_similarity(card_vecs, topic_vecs)  # (N, T)
    df["frontier_score"]       = sim_matrix.max(axis=1).round(4)
    df["best_topic_idx"]       = sim_matrix.argmax(axis=1)
    df["best_topic"]           = df["best_topic_idx"].apply(lambda i: FRONTIER_TOPICS[i][:60])

    return df


# ── Step 4: Label & signal analysis ──────────────────────────────────────────

def add_labels_and_analyse(df: pd.DataFrame) -> pd.DataFrame:
    """
    Simulate the label that would be assigned after 30 days:
    'high_adoption' = 1 if downloads in the top ADOPTION_TOP_PERCENTILE percent.
    Then print a quick signal analysis.
    """
    threshold = df["downloads"].quantile(1 - ADOPTION_TOP_PERCENTILE / 100)
    df["high_adoption"] = (df["downloads"] >= threshold).astype(int)
    return df


def print_signal_report(df: pd.DataFrame) -> None:
    """Compute and display feature-to-label correlations."""
    feature_cols = [
        "frontier_score", "age_hours", "dl_per_hour",
        "likes_per_hour", "likes", "trending_score", "weekend_upload",
    ]
    available = [c for c in feature_cols if c in df.columns]

    corr = df[available + ["high_adoption"]].corr()["high_adoption"].drop("high_adoption")
    corr_df = corr.abs().sort_values(ascending=False).reset_index()
    corr_df.columns = ["feature", "|correlation|"]
    corr_df["|correlation|"] = corr_df["|correlation|"].round(4)

    table = Table(title="Feature → Label Correlations (|Pearson|)", show_lines=True)
    table.add_column("Rank", style="dim", width=6)
    table.add_column("Feature", style="cyan")
    table.add_column("|Correlation with high_adoption|", style="green")
    table.add_column("Signal strength", style="yellow")

    for i, row in corr_df.iterrows():
        c = row["|correlation|"]
        strength = ("🔴 weak" if c < 0.05 else
                    "🟡 moderate" if c < 0.15 else
                    "🟢 strong")
        table.add_row(str(i + 1), row["feature"], f"{c:.4f}", strength)

    console.print(table)


def print_sample_models(df: pd.DataFrame, n: int = 10) -> None:
    """Print the highest frontier-score models with key stats."""
    top = df.nlargest(n, "frontier_score")[
        ["model_id", "frontier_score", "downloads", "likes",
         "dl_per_hour", "high_adoption", "best_topic"]
    ].reset_index(drop=True)

    table = Table(title=f"Top {n} Models by Frontier Similarity Score", show_lines=True)
    table.add_column("#", style="dim", width=4)
    table.add_column("Model ID", style="cyan", max_width=40)
    table.add_column("Frontier\nScore", justify="right", style="green")
    table.add_column("Downloads", justify="right")
    table.add_column("Likes", justify="right")
    table.add_column("DL/hr", justify="right")
    table.add_column("Label", justify="center")
    table.add_column("Best Matching Topic", max_width=40, style="dim")

    for i, row in top.iterrows():
        label_str = "[green]✓ HIGH[/green]" if row["high_adoption"] else "[red]✗ LOW[/red]"
        table.add_row(
            str(i + 1),
            row["model_id"],
            f"{row['frontier_score']:.3f}",
            f"{int(row['downloads']):,}",
            str(int(row["likes"])),
            f"{row['dl_per_hour']:.3f}",
            label_str,
            row["best_topic"],
        )

    console.print(table)


def print_frontier_filter_stats(df: pd.DataFrame) -> None:
    """Show how many models pass the similarity threshold."""
    above = (df["frontier_score"] >= SIMILARITY_THRESHOLD).sum()
    total = len(df)
    pct = above / total * 100 if total else 0

    rprint(Panel(
        f"[bold]Frontier filter stats[/bold]\n\n"
        f"  Total models fetched : [cyan]{total}[/cyan]\n"
        f"  Above threshold ({SIMILARITY_THRESHOLD:.2f}) : [green]{above}[/green] "
        f"([yellow]{pct:.1f}%[/yellow])\n"
        f"  Below threshold       : [red]{total - above}[/red]\n\n"
        f"[dim]These {above} models would be stored in the feature store.[/dim]",
        title="[bold blue]Step 1 — Frontier Filtering",
        border_style="blue",
    ))


def print_feasibility_verdict(df: pd.DataFrame) -> None:
    """Summarise whether the POC passes each feasibility criterion."""
    frontier_filtered = (df["frontier_score"] >= SIMILARITY_THRESHOLD).sum()
    max_corr = df[["dl_per_hour", "likes_per_hour", "trending_score",
                   "frontier_score"]].corrwith(df["high_adoption"]).abs().max()

    checks = [
        (
            "HF Hub API accessible & returns real data",
            len(df) > 0,
            f"{len(df)} models fetched with metadata"
        ),
        (
            "Embedding model loads and scores models",
            df["frontier_score"].max() > 0,
            f"Max frontier score = {df['frontier_score'].max():.3f}"
        ),
        (
            "Frontier filter produces a meaningful subset",
            5 <= frontier_filtered <= len(df),
            f"{frontier_filtered} / {len(df)} models pass threshold"
        ),
        (
            "Velocity/metadata features show signal",
            max_corr >= 0.05,
            f"Best |corr| with adoption label = {max_corr:.3f}"
        ),
        (
            "Label can be derived from existing data",
            "high_adoption" in df.columns,
            "Binary label from download percentile rank ✓"
        ),
    ]

    table = Table(title="Feasibility Verdict", show_lines=True)
    table.add_column("Criterion", style="white", max_width=50)
    table.add_column("Pass?", justify="center", width=8)
    table.add_column("Evidence", style="dim", max_width=45)

    all_pass = True
    for criterion, passed, evidence in checks:
        status = "[green]✓ YES[/green]" if passed else "[red]✗ NO[/red]"
        if not passed:
            all_pass = False
        table.add_row(criterion, status, evidence)

    console.print(table)

    if all_pass:
        rprint(Panel(
            "[bold green]✅  ALL CHECKS PASSED — Project is technically feasible.[/bold green]\n\n"
            "The HF Hub API delivers the required data, embeddings score frontier relevance\n"
            "correctly, and velocity features correlate with adoption. Safe to proceed to\n"
            "the full Feature Pipeline implementation.",
            border_style="green",
        ))
    else:
        rprint(Panel(
            "[bold red]⚠️  One or more checks FAILED — review output above.[/bold red]",
            border_style="red",
        ))


# ── Main ──────────────────────────────────────────────────────────────────────

def main() -> None:
    console.rule("[bold blue]HF Frontier Adoption Forecaster — Feasibility POC")

    # 1. Fetch
    models = fetch_models(limit=MAX_MODELS_TO_FETCH)
    if not models:
        rprint("[red]No models returned by the API. Check your network connection.[/red]")
        sys.exit(1)

    # 2. Feature extraction
    console.log("[cyan]Extracting metadata features…[/cyan]")
    df = build_features(models)
    console.log(f"[green]✓ Feature dataframe shape: {df.shape}[/green]\n")

    # 3. Embed + frontier score
    df = compute_frontier_scores(df)

    # 4. Labels + analysis
    df = add_labels_and_analyse(df)

    # ── Reports ──
    console.rule("Results")
    print_frontier_filter_stats(df)
    console.print()
    print_sample_models(df, n=10)
    console.print()
    print_signal_report(df)
    console.print()
    print_feasibility_verdict(df)

    # 5. Save snapshot
    out_path = "poc_snapshot.parquet"
    df.drop(columns=["card_text", "best_topic_idx"], errors="ignore").to_parquet(
        out_path, index=False
    )
    rprint(f"\n[dim]Raw feature snapshot saved → [bold]{out_path}[/bold][/dim]")
    console.rule()


if __name__ == "__main__":
    main()


──────────────────────────────── HF Frontier Adoption Forecaster — Feasibility POC ────────────────────────────────

[15:02:06] Querying HF Hub API for up to 200 recent frontier models…                                ]8;id=927873;file:///var/folders/vz/rlpym5yn2jq9bgt_l5m6vrz80000gn/T/ipykernel_43366/545875352.py\545875352.py]8;;\:]8;id=927874;file:///var/folders/vz/rlpym5yn2jq9bgt_l5m6vrz80000gn/T/ipykernel_43366/545875352.py#79\79]8;;\

             keyword=vla                            → 100 models                                    ]8;id=927880;file:///var/folders/vz/rlpym5yn2jq9bgt_l5m6vrz80000gn/T/ipykernel_43366/545875352.py\545875352.py]8;;\:]8;id=927881;file:///var/folders/vz/rlpym5yn2jq9bgt_l5m6vrz80000gn/T/ipykernel_43366/545875352.py#91\91]8;;\

             keyword=vision-language-action         → 100 models                                    ]8;id=927886;file:///var/folders/vz/rlpym5yn2jq9bgt_l5m6vrz80000gn/T/ipykernel_43366/545875352.py\545875352.py]8;;\:]8;id=927887;file:///var/folders/vz/rlpym5yn2jq9bgt_l5m6vrz80000gn/T/ipykernel_43366/545875352.py#91\91]8;;\

             keyword=robotics                       → 100 models                                    ]8;id=927892;file:///var/folders/vz/rlpym5yn2jq9bgt_l5m6vrz80000gn/T/ipykernel_43366/545875352.py\545875352.py]8;;\:]8;id=927893;file:///var/folders/vz/rlpym5yn2jq9bgt_l5m6vrz80000gn/T/ipykernel_43366/545875352.py#91\91]8;;\

...

                    Feature → Label Correlations (|Pearson|)                    
┏━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Rank   ┃ Feature        ┃ |Correlation with high_adoption| ┃ Signal strength ┃
┡━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ 1      │ dl_per_hour    │ 0.3522                           │ 🟢 strong       │
├────────┼────────────────┼──────────────────────────────────┼─────────────────┤
│ 2      │ likes_per_hour │ 0.3170                           │ 🟢 strong       │
├────────┼────────────────┼──────────────────────────────────┼─────────────────┤
│ 3      │ likes          │ 0.2119                           │ 🟢 strong       │
├────────┼────────────────┼──────────────────────────────────┼─────────────────┤
│ 4      │ frontier_score │ 0.1384                           │ 🟡 moderate     │
├────────┼────────────────┼──────────────────────────────────┼─────────────────┤
│ 5      │ age_hours      │ 0.0499                           │ 🔴 weak         │
├────────┼────────────────┼──────────────────────────────────┼─────────────────┤
│ 6      │ weekend_upload │ 0.0485                           │ 🔴 weak         │
├────────┼────────────────┼──────────────────────────────────┼─────────────────┤
│ 7      │ trending_score │ nan                              │ 🟢 strong       │
└────────┴────────────────┴──────────────────────────────────┴─────────────────┘

                                           Feasibility Verdict                                            
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Criterion                                    ┃  Pass?   ┃ Evidence                                     ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ HF Hub API accessible & returns real data    │  ✓ YES   │ 100 models fetched with metadata             │
├──────────────────────────────────────────────┼──────────┼──────────────────────────────────────────────┤
│ Embedding model loads and scores models      │  ✓ YES   │ Max frontier score = 0.573                   │
├──────────────────────────────────────────────┼──────────┼──────────────────────────────────────────────┤
│ Frontier filter produces a meaningful subset │  ✓ YES   │ 91 / 100 models pass threshold               │
├──────────────────────────────────────────────┼──────────┼──────────────────────────────────────────────┤
│ Velocity/metadata features show signal       │  ✓ YES   │ Best |corr| with adoption label = 0.352      │
├──────────────────────────────────────────────┼──────────┼──────────────────────────────────────────────┤
│ Label can be derived from existing data      │  ✓ YES   │ Binary label from download percentile rank ✓ │
└──────────────────────────────────────────────┴──────────┴──────────────────────────────────────────────┘

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅  ALL CHECKS PASSED — Project is technically feasible.                                                        │
│                                                                                                                 │
│ The HF Hub API delivers the required data, embeddings score frontier relevance                                  │
│ correctly, and velocity features correlate with adoption. Safe to proceed to                                    │
│ the full Feature Pipeline implementation.                                                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Raw feature snapshot saved → poc_snapshot.parquet

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────